# Miyuna — Amazon Baby Dataset Pipeline (Google Colab)

**Source:** `KAGGLE_AMAZON_BABY_ROOPALIK`  
**Real CSV:** `reviews_Baby_5_final_dataset.csv`

## Governance
- provenanceStatus = PARTIAL
- licenseStatus = UNKNOWN
- usageRightsStatus = UNKNOWN
- datasetEligibility = QUARANTINED
- trainingAllowed = NO

Training outputs are blocked.

In [ ]:
# Configuration
USE_KAGGLE_API = False     # Set True to download via Kaggle API
USER_UPLOAD = True         # Upload CSV or ZIP manually

KAGGLE_DATASET = "roopalik/amazon-baby-dataset"
KAGGLE_DOWNLOAD_DIR = "/content/amazon_baby"
OUTPUT_DIR = "/content/miyuna_output"
ADAPTER_PATH = "/content/miyuna_adapter.py"

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas"])
if USE_KAGGLE_API and not USER_UPLOAD:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kaggle"])
print("Dependencies ready.")

In [ ]:
from google.colab import files
import shutil
from pathlib import Path

adapter = Path(ADAPTER_PATH)
repo_candidates = [
    Path("/content/cocuk-urun-analiz/scripts/amazon_baby/miyuna_adapter.py"),
    Path("/content/scripts/amazon_baby/miyuna_adapter.py"),
    Path("miyuna_adapter.py"),
]

if not adapter.exists():
    copied = False
    for candidate in repo_candidates:
        if candidate.exists():
            shutil.copy(candidate, adapter)
            copied = True
            print(f"Using adapter from {candidate}")
            break
    if not copied:
        print("Upload scripts/amazon_baby/miyuna_adapter.py from the Miyuna repo:")
        uploaded = files.upload()
        if "miyuna_adapter.py" not in uploaded:
            raise FileNotFoundError("Expected uploaded file: miyuna_adapter.py")
        Path("miyuna_adapter.py").rename(adapter)

spec = importlib.util.spec_from_file_location("miyuna_adapter", adapter)
miyuna_adapter = importlib.util.module_from_spec(spec)
spec.loader.exec_module(miyuna_adapter)
print(f"Loaded adapter: {adapter}")

In [ ]:
import os
from pathlib import Path

manual_upload_path = None

if USER_UPLOAD:
    print("Upload Amazon Baby CSV or ZIP (e.g. reviews_Baby_5_final_dataset.csv/.zip):")
    uploaded = files.upload()
    upload_name = next(iter(uploaded))
    manual_upload_path = Path(f"/content/{upload_name}")
    print(f"Uploaded: {manual_upload_path}")
elif USE_KAGGLE_API:
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    kaggle_json = kaggle_dir / "kaggle.json"
    if not kaggle_json.exists():
        print("Upload your Kaggle API credential (kaggle.json). Credentials are NOT printed.")
        uploaded = files.upload()
        if "kaggle.json" not in uploaded:
            raise FileNotFoundError("Expected uploaded file: kaggle.json")
        kaggle_json.write_bytes(uploaded["kaggle.json"])
    os.chmod(kaggle_json, 0o600)
    download_dir = Path(KAGGLE_DOWNLOAD_DIR)
    download_dir.mkdir(parents=True, exist_ok=True)
    subprocess.check_call([
        "kaggle", "datasets", "download",
        "-d", KAGGLE_DATASET,
        "-p", str(download_dir),
        "--unzip",
    ])
    print(f"Kaggle dataset downloaded to {download_dir}")
else:
    raise ValueError("Enable USE_KAGGLE_API or USER_UPLOAD.")

In [ ]:
import pandas as pd

csv_path = miyuna_adapter.find_kaggle_csv(manual_path=manual_upload_path)
print(f"Using CSV: {csv_path}")

df = pd.read_csv(csv_path)
schema = miyuna_adapter.detect_schema(df)
print(f"Detected schema: {schema.kind}")

accepted, rejected, report = miyuna_adapter.adapt_dataframe(df)
paths = miyuna_adapter.write_outputs(Path(OUTPUT_DIR), accepted, rejected, report)

miyuna_adapter.print_colab_summary(report)
print(f"Import JSONL: {paths['import']}")
print(f"Quality report: {paths['report']}")

In [ ]:
ok, errors, record_count = miyuna_adapter.validate_import_jsonl(paths["import"])
print(f"INPUT_RECORD_COUNT: {record_count}")

if ok:
    print("MIYUNA_COLAB_DATASET_PREP: PASS")
else:
    print("MIYUNA_COLAB_DATASET_PREP: FAIL")
    for err in errors:
        print(f"  - {err}")
    raise RuntimeError("Output validation failed")

if not miyuna_adapter.TRAINING_RIGHTS_APPROVED:
    print("TRAINING BLOCKED: license / usage rights not approved.")

In [ ]:
from google.colab import files
import zipfile

out = Path(OUTPUT_DIR)
zip_path = out / "miyuna_amazon_baby_outputs.zip"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for name in [
        "miyuna_amazon_baby_import.jsonl",
        "miyuna_amazon_baby_quarantined.jsonl",
        "dataset_quality_report.json",
        "rejected_records.jsonl",
    ]:
        fp = out / name
        if fp.exists():
            zf.write(fp, arcname=name)

print("Downloading primary artifacts...")
files.download(str(paths["import"]))
files.download(str(paths["report"]))
files.download(str(zip_path))
print("Downloads started.")

## Copy artifact to Miyuna project (Windows)

```
C:\Users\MOSTER\Documents\GitHub\cocuk-urun-analiz\scripts\amazon_baby\output\miyuna_amazon_baby_import.jsonl
```

```powershell
Test-Path "C:\Users\MOSTER\Documents\GitHub\cocuk-urun-analiz\scripts\amazon_baby\output\miyuna_amazon_baby_import.jsonl"
```

Expected: `True`

Then resume in Cursor:

```
AMAZON_BABY_REAL_IMPORT_E2E
```